In [ ]:
# %% [markdown]
# # Play Store Review Analysis System

# %% [markdown]
# ### Step 1: Install Required Libraries

# %%
!pip install google-play-scraper textblob vaderSentiment pandas numpy matplotlib seaborn wordcloud plotly scikit-learn nltk -q
!pip install transformers -q
!python -m textblob.download_corpora

# %% [markdown]
# ### Step 2: Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-poned')

# Sentiment Analysis Libraries
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Play Store Scraper
from google_play_scraper import app, reviews, Sort

# For advanced analysis
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# %% [markdown]
# ### Step 3: Initialize Analyzer Class

# %%
class PlayStoreAnalyzer:
    def __init__(self, app_id, language='en', country='us'):
        self.app_id = app_id
        self.language = language
        self.country = country
        self.reviews_df = None
        self.app_info = None
        self.analyzed_data = None

        # Initialize sentiment analyzers
        self.vader = SentimentIntensityAnalyzer()

        # Define issue categories
        self.issue_categories = {
            'bug_issues': ['crash', 'bug', 'error', 'not working', 'freeze', 'glitch', 'broken'],
            'performance': ['slow', 'lag', 'loading', 'delay', 'unresponsive', 'hangs'],
            'ui_ux': ['interface', 'design', 'layout', 'confusing', 'ugly', 'complicated', 'navigation'],
            'feature_request': ['add', 'should have', 'need', 'missing', 'want', 'please add', 'suggestion'],
            'battery': ['battery', 'drain', 'consumption', 'power', 'charge'],
            'ads': ['ads', 'advertisement', 'popup', 'commercial', 'sponsored'],
            'update': ['update', 'version', 'new version', 'latest'],
            'login': ['login', 'sign in', 'password', 'account', 'authentication'],
            'notification': ['notification', 'alert', 'reminder'],
            'privacy': ['privacy', 'data', 'security', 'permission', 'tracking']
        }

        # Preprocessors
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def fetch_app_info(self):
        """Fetch app information from Play Store"""
        try:
            self.app_info = app(
                self.app_id,
                lang=self.language,
                country=self.country
            )
            print(f" App found: {self.app_info['title']}")
            print(f" Current rating: {self.app_info['score']} ⭐")
            print(f" Installs: {self.app_info['installs']}")
            return True
        except Exception as e:
            print(f" Error fetching app info: {e}")
            return False

    def fetch_reviews(self, count=2000, filter_score=None):
        """Fetch reviews from Play Store"""
        try:
            print(f" Fetching {count} reviews...")

            # Fetch reviews in batches
            all_reviews = []
            token = None

            for _ in range(0, count, 200):
                batch_size = min(200, count - len(all_reviews))

                result, token = reviews(
                    self.app_id,
                    lang=self.language,
                    country=self.country,
                    sort=Sort.NEWEST,
                    count=batch_size,
                    continuation_token=token
                )

                if filter_score:
                    result = [r for r in result if r['score'] == filter_score]

                all_reviews.extend(result)

                if not token:
                    break

            # Convert to DataFrame
            self.reviews_df = pd.DataFrame(all_reviews)

            if not self.reviews_df.empty:
                # Convert datetime
                self.reviews_df['at'] = pd.to_datetime(self.reviews_df['at'])

                print(f" Successfully fetched {len(self.reviews_df)} reviews")
                print(f" Date range: {self.reviews_df['at'].min().date()} to {self.reviews_df['at'].max().date()}")

                # Calculate average rating
                avg_rating = self.reviews_df['score'].mean()
                print(f" Average rating in fetched reviews: {avg_rating:.2f} ⭐")
            else:
                print(" No reviews fetched")

            return len(self.reviews_df)

        except Exception as e:
            print(f" Error fetching reviews: {e}")
            return 0

    def preprocess_text(self, text):
        """Clean and preprocess review text"""
        if not isinstance(text, str):
            return ""

        # Convert to lowercase
        text = text.lower()

        # Remove URLs, special characters, and numbers
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        # Tokenize
        tokens = word_tokenize(text)

        # Remove stopwords and lemmatize
        tokens = [self.lemmatizer.lemmatize(word) for word in tokens
                 if word not in self.stop_words and len(word) > 2]

        return ' '.join(tokens)

    def analyze_sentiment(self):
        """Perform multi-model sentiment analysis"""
        if self.reviews_df is None or self.reviews_df.empty:
            print(" No reviews to analyze")
            return

        print(" Performing sentiment analysis...")

        # Preprocess text
        self.reviews_df['cleaned_content'] = self.reviews_df['content'].apply(self.preprocess_text)

        # Initialize sentiment columns
        sentiments = []
        vader_scores = []
        textblob_scores = []
        compound_scores = []

        for content in self.reviews_df['content']:
            # VADER Analysis
            vader_score = self.vader.polarity_scores(content)['compound']

            # TextBlob Analysis
            blob = TextBlob(content)
            textblob_score = blob.sentiment.polarity

            # Combined score
            compound = (vader_score + textblob_score) / 2

            # Classify sentiment
            if compound >= 0.05:
                sentiment = 'Positive'
            elif compound <= -0.05:
                sentiment = 'Negative'
            else:
                sentiment = 'Neutral'

            sentiments.append(sentiment)
            vader_scores.append(vader_score)
            textblob_scores.append(textblob_score)
            compound_scores.append(compound)

        # Add to DataFrame
        self.reviews_df['sentiment'] = sentiments
        self.reviews_df['vader_score'] = vader_scores
        self.reviews_df['textblob_score'] = textblob_scores
        self.reviews_df['compound_score'] = compound_scores

        print(f" Sentiment Analysis Complete:")
        print(f"   Positive: {len(self.reviews_df[self.reviews_df['sentiment'] == 'Positive'])} reviews")
        print(f"   Negative: {len(self.reviews_df[self.reviews_df['sentiment'] == 'Negative'])} reviews")
        print(f"   Neutral: {len(self.reviews_df[self.reviews_df['sentiment'] == 'Neutral'])} reviews")

    def categorize_issues(self):
        """Categorize reviews into issue types"""
        if self.reviews_df is None or self.reviews_df.empty:
            return

        print(" Categorizing issues...")

        # Initialize issue columns
        issue_categories_list = []

        for review in self.reviews_df['content']:
            review_lower = str(review).lower()
            categories_found = []

            for category, keywords in self.issue_categories.items():
                if any(keyword in review_lower for keyword in keywords):
                    categories_found.append(category)

            issue_categories_list.append(categories_found)

        self.reviews_df['issue_categories'] = issue_categories_list

        # Count issues
        all_issues = []
        for issues in issue_categories_list:
            all_issues.extend(issues)

        issue_counts = Counter(all_issues)
        print(f" Top Issues Found:")
        for issue, count in issue_counts.most_common(5):
            print(f"   {issue}: {count} mentions")

    def extract_keywords(self):
        """Extract most frequent keywords from reviews"""
        if self.reviews_df is None or self.reviews_df.empty:
            return

        print(" Extracting keywords...")

        # Combine all cleaned content
        all_text = ' '.join(self.reviews_df['cleaned_content'].tolist())

        # Split into words and count frequencies
        words = all_text.split()
        word_freq = Counter(words)

        # Get most common words (excluding very common ones)
        common_words = set(['app', 'like', 'good', 'great', 'use', 'would', 'get', 'one'])
        filtered_freq = {word: count for word, count in word_freq.items()
                        if word not in common_words and count > 5}

        # Sort by frequency
        self.top_keywords = sorted(filtered_freq.items(), key=lambda x: x[1], reverse=True)[:20]

        print(f" Top 5 Keywords:")
        for word, count in self.top_keywords[:5]:
            print(f"   '{word}': {count} occurrences")

    def analyze_rating_trends(self):
        """Analyze rating trends over time"""
        if self.reviews_df is None or self.reviews_df.empty:
            return

        print(" Analyzing rating trends...")

        # Group by date
        self.reviews_df['date'] = self.reviews_df['at'].dt.date
        daily_stats = self.reviews_df.groupby('date').agg({
            'score': 'mean',
            'content': 'count',
            'compound_score': 'mean'
        }).reset_index()

        daily_stats.columns = ['date', 'avg_rating', 'review_count', 'avg_sentiment']

        # Calculate moving averages
        daily_stats['rating_ma'] = daily_stats['avg_rating'].rolling(window=7, min_periods=1).mean()
        daily_stats['sentiment_ma'] = daily_stats['avg_sentiment'].rolling(window=7, min_periods=1).mean()

        self.daily_stats = daily_stats
        print(f" Trend analysis complete for {len(daily_stats)} days")

    def generate_insights(self):
        """Generate comprehensive insights"""
        if self.reviews_df is None or self.reviews_df.empty:
            return

        print(" Generating insights...")

        insights = {
            'overall_sentiment': {
                'positive_pct': len(self.reviews_df[self.reviews_df['sentiment'] == 'Positive']) / len(self.reviews_df) * 100,
                'negative_pct': len(self.reviews_df[self.reviews_df['sentiment'] == 'Negative']) / len(self.reviews_df) * 100,
                'neutral_pct': len(self.reviews_df[self.reviews_df['sentiment'] == 'Neutral']) / len(self.reviews_df) * 100,
            },
            'rating_stats': {
                'avg_rating': self.reviews_df['score'].mean(),
                'median_rating': self.reviews_df['score'].median(),
                'rating_distribution': self.reviews_df['score'].value_counts().sort_index().to_dict()
            },
            'common_issues': {},
            'user_impact': {}
        }

        # Analyze issue distribution
        all_issues = []
        for issues in self.reviews_df['issue_categories']:
            all_issues.extend(issues)

        issue_counts = Counter(all_issues)
        insights['common_issues'] = dict(issue_counts.most_common(10))

        # Extract sample impactful reviews
        positive_impact = self.reviews_df[self.reviews_df['sentiment'] == 'Positive'].nlargest(3, 'compound_score')['content'].tolist()
        negative_impact = self.reviews_df[self.reviews_df['sentiment'] == 'Negative'].nsmallest(3, 'compound_score')['content'].tolist()

        insights['user_impact'] = {
            'positive_examples': positive_impact,
            'negative_examples': negative_impact
        }

        # Sentiment by rating
        sentiment_by_rating = self.reviews_df.groupby('score')['sentiment'].value_counts(normalize=True).unstack().fillna(0)
        insights['sentiment_by_rating'] = sentiment_by_rating.to_dict()

        self.insights = insights
        return insights

    def visualize_results(self):
        """Create comprehensive visualizations"""
        if self.reviews_df is None or self.reviews_df.empty:
            print(" No data to visualize")
            return

        print(" Creating visualizations...")

        # Set style
        plt.style.use('seaborn-v0_8-darkgrid')
        sns.set_palette("husl")

        # Create subplots
        fig = plt.figure(figsize=(20, 15))

        # 1. Sentiment Distribution
        plt.subplot(3, 3, 1)
        sentiment_counts = self.reviews_df['sentiment'].value_counts()
        plt.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
                colors=['#4CAF50', '#FF5252', '#FFC107'])
        plt.title('Sentiment Distribution', fontsize=14, fontweight='bold')

        # 2. Rating Distribution
        plt.subplot(3, 3, 2)
        rating_counts = self.reviews_df['score'].value_counts().sort_index()
        bars = plt.bar(rating_counts.index.astype(str), rating_counts.values,
                      color=['#FF5252', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50'])
        plt.title('Star Rating Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Stars')
        plt.ylabel('Count')

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom')

        # 3. Word Cloud
        plt.subplot(3, 3, 3)
        all_text = ' '.join(self.reviews_df['cleaned_content'].tolist())
        wordcloud = WordCloud(width=800, height=400, background_color='white',
                             max_words=100, contour_width=3, contour_color='steelblue').generate(all_text)
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Most Frequent Words', fontsize=14, fontweight='bold')

        # 4. Sentiment by Rating
        plt.subplot(3, 3, 4)
        sentiment_by_rating = pd.crosstab(self.reviews_df['score'], self.reviews_df['sentiment'])
        sentiment_by_rating.plot(kind='bar', stacked=True, ax=plt.gca())
        plt.title('Sentiment by Star Rating', fontsize=14, fontweight='bold')
        plt.xlabel('Stars')
        plt.ylabel('Count')
        plt.legend(title='Sentiment')

        # 5. Issue Categories
        plt.subplot(3, 3, 5)
        all_issues = []
        for issues in self.reviews_df['issue_categories']:
            all_issues.extend(issues)

        issue_counts = pd.Series(all_issues).value_counts().head(8)
        issue_counts.plot(kind='barh', color='coral')
        plt.title('Top Issue Categories', fontsize=14, fontweight='bold')
        plt.xlabel('Count')

        # 6. Rating Trend
        if hasattr(self, 'daily_stats') and self.daily_stats is not None:
            plt.subplot(3, 3, 6)
            plt.plot(self.daily_stats['date'], self.daily_stats['rating_ma'],
                    linewidth=2, color='blue', marker='o', markersize=3)
            plt.title('7-Day Moving Average of Ratings', fontsize=14, fontweight='bold')
            plt.xlabel('Date')
            plt.ylabel('Average Rating')
            plt.xticks(rotation=45)
            plt.grid(True, alpha=0.3)

        # 7. Review Length Analysis
        plt.subplot(3, 3, 7)
        self.reviews_df['review_length'] = self.reviews_df['content'].apply(lambda x: len(str(x).split()))
        plt.hist(self.reviews_df['review_length'], bins=30, edgecolor='black', alpha=0.7)
        plt.title('Review Length Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Number of Words')
        plt.ylabel('Frequency')

        # 8. Sentiment Score Distribution
        plt.subplot(3, 3, 8)
        plt.hist(self.reviews_df['compound_score'], bins=30, edgecolor='black', alpha=0.7, color='purple')
        plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
        plt.title('Sentiment Score Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Compound Sentiment Score')
        plt.ylabel('Frequency')

        # 9. Correlation between rating and sentiment
        plt.subplot(3, 3, 9)
        plt.scatter(self.reviews_df['score'], self.reviews_df['compound_score'],
                   alpha=0.5, s=20, c=self.reviews_df['compound_score'], cmap='coolwarm')
        plt.colorbar(label='Sentiment Score')
        plt.title('Rating vs Sentiment Score', fontsize=14, fontweight='bold')
        plt.xlabel('Star Rating')
        plt.ylabel('Sentiment Score')

        plt.tight_layout()
        plt.savefig('analysis_results.png', dpi=300, bbox_inches='tight')
        plt.show()

        print(" Visualizations saved as 'analysis_results.png'")

    def generate_text_report(self, app_name):
        """Generate comprehensive text report"""
        if not hasattr(self, 'insights'):
            self.generate_insights()

        report = []
        report.append("=" * 70)
        report.append(f" PLAY STORE REVIEW ANALYSIS REPORT")
        report.append(f" App: {app_name}")
        report.append(f" Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append(f" Total Reviews Analyzed: {len(self.reviews_df)}")
        report.append("=" * 70)
        report.append("")

        # App Information
        if self.app_info:
            report.append(" APP INFORMATION:")
            report.append(f"   • Title: {self.app_info.get('title', 'N/A')}")
            report.append(f"   • Current Rating: {self.app_info.get('score', 'N/A')} ⭐")
            report.append(f"   • Installs: {self.app_info.get('installs', 'N/A')}")
            report.append(f"   • Developer: {self.app_info.get('developer', 'N/A')}")
            report.append(f"   • Category: {self.app_info.get('genre', 'N/A')}")
            report.append("")

        # Sentiment Analysis
        report.append(" SENTIMENT ANALYSIS:")
        report.append(f"   • Positive Reviews: {self.insights['overall_sentiment']['positive_pct']:.1f}%")
        report.append(f"   • Negative Reviews: {self.insights['overall_sentiment']['negative_pct']:.1f}%")
        report.append(f"   • Neutral Reviews: {self.insights['overall_sentiment']['neutral_pct']:.1f}%")
        report.append("")

        # Rating Statistics
        report.append(" RATING STATISTICS:")
        report.append(f"   • Average Rating: {self.insights['rating_stats']['avg_rating']:.2f} ⭐")
        report.append(f"   • Median Rating: {self.insights['rating_stats']['median_rating']:.2f} ⭐")
        report.append("   • Rating Distribution:")
        for stars, count in sorted(self.insights['rating_stats']['rating_distribution'].items()):
            percentage = (count / len(self.reviews_df)) * 100
            report.append(f"     {int(stars)} stars: {count} reviews ({percentage:.1f}%)")
        report.append("")

        # Top Issues
        if self.insights['common_issues']:
            report.append(" TOP ISSUES IDENTIFIED:")
            for issue, count in self.insights['common_issues'].items():
                report.append(f"   • {issue.replace('_', ' ').title()}: {count} mentions")
            report.append("")

        # User Impact Examples
        report.append(" USER IMPACT EXAMPLES:")
        report.append("   Positive Impact Reviews:")
        for i, review in enumerate(self.insights['user_impact']['positive_examples'][:2], 1):
            review_text = review[:150] + "..." if len(review) > 150 else review
            report.append(f"     {i}. \"{review_text}\"")

        report.append("")
        report.append("   Negative Impact Reviews:")
        for i, review in enumerate(self.insights['user_impact']['negative_examples'][:2], 1):
            review_text = review[:150] + "..." if len(review) > 150 else review
            report.append(f"     {i}. \"{review_text}\"")
        report.append("")

        # Recommendations
        report.append(" RECOMMENDATIONS:")

        # Based on sentiment
        if self.insights['overall_sentiment']['negative_pct'] > 30:
            report.append("   •  High negative sentiment detected! Prioritize issue resolution.")

        # Based on issues
        if 'bug_issues' in self.insights['common_issues']:
            report.append("   •  Fix critical bugs and crashes immediately")

        if 'performance' in self.insights['common_issues']:
            report.append("   •  Optimize app performance and reduce lag")

        if 'ads' in self.insights['common_issues']:
            report.append("   •  Reconsider ad frequency or add ad-free option")

        if 'feature_request' in self.insights['common_issues']:
            report.append("   •  Implement most requested features")

        if 'battery' in self.insights['common_issues']:
            report.append("   •  Optimize battery consumption")

        report.append("")
        report.append("=" * 70)

        # Save report to file
        report_text = '\n'.join(report)
        with open(f'{app_name.replace(" ", "_")}_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write(report_text)

        # Print report
        print(report_text)
        print(f"\n Report saved as '{app_name.replace(' ', '_')}_analysis_report.txt'")

        return report_text

    def full_analysis(self, app_name, review_count=1000):
        """Run complete analysis pipeline"""
        print(f"\n Starting analysis for: {app_name}")
        print("=" * 50)

        # Fetch app info
        self.fetch_app_info()

        # Fetch reviews
        self.fetch_reviews(count=review_count)

        if self.reviews_df is None or self.reviews_df.empty:
            print(" No reviews to analyze. Exiting.")
            return None

        # Run analysis
        self.analyze_sentiment()
        self.categorize_issues()
        self.extract_keywords()
        self.analyze_rating_trends()

        # Generate insights
        insights = self.generate_insights()

        # Visualize
        self.visualize_results()

        # Generate report
        report = self.generate_text_report(app_name)

        return insights

# %% [markdown]
# ### Step 4: Initialize and Run Analysis

# %%
# @title Enter App Details
app_name = "adobe"  # @param {type:"string"}
app_id = "com.adobe.scan.android"  # @param {type:"string"}
review_count = 5000  # @param {type:"slider", min:100, max:5000, step:100}

# Initialize analyzer
analyzer = PlayStoreAnalyzer(app_id)

# Run full analysis
insights = analyzer.full_analysis(app_name, review_count)

# %% [markdown]
# ### Step 5: Display Detailed Statistics

# %%
if analyzer.reviews_df is not None and not analyzer.reviews_df.empty:
    print("\n DETAILED STATISTICS:")
    print("=" * 40)

    # Display DataFrame info
    print("\n Reviews DataFrame Info:")
    analyzer.reviews_df.info()

    print("\n Summary Statistics:")
    print(analyzer.reviews_df[['score', 'vader_score', 'textblob_score', 'compound_score']].describe())

    print("\n Reviews by Month:")
    analyzer.reviews_df['month'] = analyzer.reviews_df['at'].dt.to_period('M')
    monthly_counts = analyzer.reviews_df['month'].value_counts().sort_index()
    print(monthly_counts.head(12))

    # Show sample reviews
    print("\n📝 Sample Reviews:")
    print("\nTop Positive Reviews:")
    for i, (_, row) in enumerate(analyzer.reviews_df.nlargest(3, 'compound_score').iterrows(), 1):
        print(f"{i}. {row['score']} - {row['content'][:150]}...")

    print("\nTop Negative Reviews:")
    for i, (_, row) in enumerate(analyzer.reviews_df.nsmallest(3, 'compound_score').iterrows(), 1):
        print(f"{i}. {row['score']} - {row['content'][:150]}...")

# %% [markdown]
# ### Step 6: Export Data

# %%
if analyzer.reviews_df is not None and not analyzer.reviews_df.empty:
    # Save analyzed data
    analyzer.reviews_df.to_csv(f'{app_name.replace(" ", "_")}_analyzed_reviews.csv', index=False, encoding='utf-8')

    # Save insights as JSON
    import json
    with open(f'{app_name.replace(" ", "_")}_insights.json', 'w') as f:
        json.dump(analyzer.insights, f, indent=2, default=str)

    print("\n DATA EXPORT COMPLETE:")
    print(f"   • CSV File: {app_name.replace(' ', '_')}_analyzed_reviews.csv")
    print(f"   • JSON Insights: {app_name.replace(' ', '_')}_insights.json")
    print(f"   • Report: {app_name.replace(' ', '_')}_analysis_report.txt")
    print(f"   • Visualization: analysis_results.png")

# %% [markdown]
# ### Step 7: Advanced Analysis - Topic Modeling (Optional)

# %%
# @title Advanced Analysis: Topic Modeling
def perform_topic_modeling(reviews_df, num_topics=5):
    """Perform basic topic modeling on reviews"""
    from sklearn.decomposition import LatentDirichletAllocation
    from sklearn.feature_extraction.text import CountVectorizer

    print(" Performing Topic Modeling...")

    # Prepare text data
    documents = reviews_df['cleaned_content'].tolist()

    # Create document-term matrix
    vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
    doc_term_matrix = vectorizer.fit_transform(documents)

    # Fit LDA model
    lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda.fit(doc_term_matrix)

    # Display topics
    feature_names = vectorizer.get_feature_names_out()

    print(f"\n Discovered Topics:")
    for topic_idx, topic in enumerate(lda.components_):
        print(f"\nTopic #{topic_idx + 1}:")
        top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
        print(f"   Top words: {', '.join(top_words)}")

    # Assign dominant topic to each review
    topic_results = lda.transform(doc_term_matrix)
    reviews_df['dominant_topic'] = topic_results.argmax(axis=1)

    return lda, reviews_df

# Run topic modeling if we have reviews
if analyzer.reviews_df is not None and len(analyzer.reviews_df) > 50:
    perform_topic_modeling(analyzer.reviews_df)
else:
    print(" Not enough reviews for topic modeling")

# %% [markdown]
# ### Step 8: Create Interactive Dashboard

# %%
# @title Interactive Dashboard
import plotly.express as px

if analyzer.reviews_df is not None and not analyzer.reviews_df.empty:
    # Create interactive plots
    print(" Creating Interactive Dashboard...")

    # 1. Interactive sentiment over time
    if hasattr(analyzer, 'daily_stats'):
        fig1 = px.line(analyzer.daily_stats, x='date', y=['avg_rating', 'avg_sentiment'],
                      title='Rating & Sentiment Trends Over Time',
                      labels={'value': 'Score', 'variable': 'Metric'})
        fig1.show()

    # 2. Interactive sunburst chart for sentiment by rating
    sentiment_by_rating = analyzer.reviews_df.groupby(['score', 'sentiment']).size().reset_index(name='count')
    fig2 = px.sunburst(sentiment_by_rating, path=['score', 'sentiment'], values='count',
                      title='Sentiment Distribution by Star Rating',
                      color='score', color_continuous_scale='RdYlGn')
    fig2.show()

    # 3. Interactive box plot of sentiment scores by rating
    fig3 = px.box(analyzer.reviews_df, x='score', y='compound_score',
                 title='Sentiment Score Distribution by Star Rating',
                 points='all')
    fig3.show()

    print(" Interactive dashboard created!")

# %% [markdown]
# ### Step 9: Analysis for Multiple Apps (Comparison)

# %%
# @title Compare Multiple Apps
def compare_apps(app_list, review_count=500):
    """Compare multiple apps"""
    comparison_results = {}

    for app_name, app_id in app_list.items():
        print(f"\n Analyzing {app_name}...")
        analyzer = PlayStoreAnalyzer(app_id)
        analyzer.fetch_app_info()
        analyzer.fetch_reviews(count=review_count)

        if analyzer.reviews_df is not None and not analyzer.reviews_df.empty:
            analyzer.analyze_sentiment()
            analyzer.generate_insights()

            comparison_results[app_name] = {
                'avg_rating': analyzer.insights['rating_stats']['avg_rating'],
                'positive_pct': analyzer.insights['overall_sentiment']['positive_pct'],
                'review_count': len(analyzer.reviews_df),
                'top_issues': list(analyzer.insights['common_issues'].keys())[:3]
            }

    # Create comparison DataFrame
    comparison_df = pd.DataFrame(comparison_results).T

    # Visualize comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. Average Rating Comparison
    axes[0, 0].bar(comparison_df.index, comparison_df['avg_rating'])
    axes[0, 0].set_title('Average Rating Comparison')
    axes[0, 0].set_ylabel('Stars')
    axes[0, 0].tick_params(axis='x', rotation=45)

    # 2. Positive Sentiment Comparison
    axes[0, 1].bar(comparison_df.index, comparison_df['positive_pct'])
    axes[0, 1].set_title('Positive Sentiment Percentage')
    axes[0, 1].set_ylabel('Percentage (%)')
    axes[0, 1].tick_params(axis='x', rotation=45)

    # 3. Review Count Comparison
    axes[1, 0].bar(comparison_df.index, comparison_df['review_count'])
    axes[1, 0].set_title('Number of Reviews Analyzed')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].tick_params(axis='x', rotation=45)

    axes[1, 1].axis('off')
    axes[1, 1].text(0.1, 0.5, 'Top Issues Comparison:\n\n' +
                   '\n'.join([f"{app}: {', '.join(issues)}"
                            for app, issues in comparison_df['top_issues'].items()]),
                   fontsize=10, verticalalignment='center')

    plt.tight_layout()
    plt.show()

    return comparison_df

# Uncomment to run comparison (choose your apps)
# app_list = {
#     "Instagram": "com.instagram.android",
#     "Facebook": "com.facebook.katana",
#     "Twitter": "com.twitter.android",
#     "TikTok": "com.zhiliaoapp.musically"
# }
# comparison_df = compare_apps(app_list, review_count=300)

# %% [markdown]
# ### Step 10: Real-time Monitoring Setup

# %%
# @title Real-time Monitoring Setup
class ReviewMonitor:
    """Monitor reviews in real-time"""

    def __init__(self, app_id, check_interval_hours=24):
        self.app_id = app_id
        self.check_interval = check_interval_hours
        self.last_check = None
        self.analyzer = PlayStoreAnalyzer(app_id)

    def check_for_issues(self, threshold=0.3):
        """Check for sudden drop in ratings"""
        print(f" Checking for issues in {self.app_id}...")

        # Fetch recent reviews
        self.analyzer.fetch_reviews(count=200)

        if self.analyzer.reviews_df is None or len(self.analyzer.reviews_df) < 50:
            print(" Not enough reviews for monitoring")
            return

        # Analyze sentiment
        self.analyzer.analyze_sentiment()
        negative_pct = (len(self.analyzer.reviews_df[self.analyzer.reviews_df['sentiment'] == 'Negative']) /
                       len(self.analyzer.reviews_df))

        avg_rating = self.analyzer.reviews_df['score'].mean()

        # Check thresholds
        alerts = []
        if negative_pct > threshold:
            alerts.append(f" High negative sentiment: {negative_pct:.1%}")

        if avg_rating < 3.0:
            alerts.append(f" Low average rating: {avg_rating:.1f} stars")

        # Check for sudden increase in specific issues
        self.analyzer.categorize_issues()

        if alerts:
            print("\n ALERTS:")
            for alert in alerts:
                print(f"   {alert}")
        else:
            print(" No critical issues detected")

        return alerts

# Initialize monitor (example)
# monitor = ReviewMonitor("com.instagram.android")
# alerts = monitor.check_for_issues()

# %% [markdown]
# ## Summary
#
# This complete Play Store Review Analysis System provides:
#
# 1. ** Data Collection**: Fetches reviews from Play Store
# 2. ** Sentiment Analysis**: Uses VADER + TextBlob for accurate sentiment
# 3. ** Issue Categorization**: Automatically categorizes common problems
# 4. ** Trend Analysis**: Tracks ratings and sentiment over time
# 5. ** Visualization**: Creates comprehensive charts and graphs
# 6. ** Report Generation**: Produces detailed text reports
# 7. ** Data Export**: Saves results in CSV, JSON, and image formats
# 8. ** Advanced Features**: Topic modeling, comparison, and monitoring
#
# ### To run for any app:
# 1. Change the `app_id` variable (Find it in Play Store URL)
# 2. Adjust `review_count` as needed
# 3. Run all cells sequentially
#
# ### Example App IDs:
# - Instagram: `com.instagram.android`
# - Facebook: `com.facebook.katana`
# - WhatsApp: `com.whatsapp`
# - YouTube: `com.google.android.youtube`
# - TikTok: `com.zhiliaoapp.musically`

# %% [markdown]
# ### Troubleshooting:
#
# If you encounter errors:
# 1. **Rate limiting**: Wait a few minutes and try again
# 2. **App not found**: Check the app ID is correct
# 3. **No reviews**: Try different language/country settings
# 4. **Memory issues**: Reduce `review_count`